### Question/Answer apllication with GraphDB

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE=os.getenv("NEO4J_DATABASE")
AURA_INSTANCEID=os.getenv("AURA_INSTANCEID")
AURA_INSTANCENAME=os.getenv("AURA_INSTANCENAME")

In [28]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(
    url=NEO4J_URI, 
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD, 
    database=NEO4J_DATABASE
)
graph

In [13]:
movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv' AS row
MERGE (m:Movie {id: row.movieId})
SET m.released = row.released,
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:GENRE]->(g))
"""

In [14]:
graph.query(movie_query)

[]

In [15]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: STRING, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [16]:
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [41]:
## Creating function to clean the response from the model:"qwen/qwen3-32b" and strip the <think> ....< /think> tags from the response
import re
def strip_thinking_process(llm_output):
    # If the output is a string, strip the tags
    if isinstance(llm_output, str):
        return re.sub(r'<think>.*?</think>', '', llm_output, flags=re.DOTALL).strip()
    # If it's a message object (standard for ChatModels)
    if hasattr(llm_output, "content"):
        llm_output.content = re.sub(r'<think>.*?</think>', '', llm_output.content, flags=re.DOTALL).strip()
    return llm_output

In [42]:
## Initialize the ChatGroq model
llm = ChatGroq(model="qwen/qwen3-32b")

from langchain_core.runnables import RunnableLambda

llm_clean=llm | RunnableLambda(strip_thinking_process)

In [43]:
from langchain_classic.chains import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm_clean,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x7c00d832c550>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_out

In [50]:
response = chain.invoke({"query": "Tell me the list of movies, where the actor 'Tom Hanks' is acted? Consider the title as movie."})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WHERE p.name = 'Tom Hanks' RETURN m.title AS movie
Full Context:
[{'movie': 'Toy Story'}, {'movie': 'Apollo 13'}]

> Finished chain.


{'query': "Tell me the list of movies, where the actor 'Tom Hanks' is acted? Consider the title as movie.",
 'result': 'Toy Story, Apollo 13'}